# Lezione 03 — Preprocessing linguistico

**Corso:** NLP Classico con Python · **Blocco 1:** Dal testo ai token
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/03_preprocesamiento_linguistico_IT.md`

## Cosa vedrai in questa lezione

- **Stopword con criterio**: cosa contengono le liste di NLTK in spagnolo e in italiano (anche negazioni!), quando aiutano e quando cambiano il significato di una frase.
- **Stemming** con Snowball: tagliare per regole, con sovra-stemming e sotto-stemming.
- **Lemmatizzazione** con spaCy: il lemma dipende dalla categoria grammaticale (POS, qui solo un'introduzione minima; si studia a fondo nella lezione 7) e i modelli piccoli sbagliano in modi prevedibili.
- **Confronto NLTK contro spaCy** in spagnolo e in italiano.
- **Un esperimento guidato**: come cambia la dimensione del vocabolario (tipi) dopo ogni passo, su recensioni e ticket, nelle due lingue.
- Una **guida alla decisione**: quale preprocessing per quale compito.

## Prerequisiti

- La lezione 1 (che cos'è l'NLP) e la lezione 2 (pulizia e tokenizzazione): qui tokenizziamo con `re.findall(r"\w+", ...)`.
- Librerie: `nltk`, `spacy` (con `es_core_news_sm` e `it_core_news_sm`), `pandas`, `matplotlib`. La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso, accanto al notebook o in `/content/datos/` (in Colab, caricala dal pannello dei file).

> **Come usare questo notebook:** esegui le celle in ordine. I corpora sono sintetici e piccoli (*esempio illustrativo*): le cifre mostrano uno schema, non sono generalizzabili. Gli output di spaCy dipendono dalla versione (qui, spaCy 3.8 e modelli 3.8.0) e possono variare leggermente con altre.

---
## Installazione e risorse

Questa cella installa ciò che manca, controlla le liste di stopword di NLTK e carica i due modelli di spaCy. Se hai già tutto, non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("nltk", "nltk")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import nltk
import spacy

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

def cargar_modelo(nombre):
    """Carica un modello di spaCy e lo scarica se non è installato."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Risorse di NLTK e modelli di spaCy pronti.")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme** e definiamo tre cose che useremo di continuo: `cargar_datos` (cerca la cartella `datos/`), `tokenizar` (quella della lezione 2) e le risorse per lingua: liste di stopword (`SW`), stemmer Snowball (`STEMMER`) e modelli di spaCy (`NLP`). Le lingue si indicano con `"spa"` (spagnolo) e `"ita"` (italiano).

In [ ]:
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

from pathlib import Path

def cargar_datos(nombre):
    """Cerca un file in datos/, ../datos/ e /content/datos/ e ne restituisce il percorso."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = Path(base) / nombre
        if ruta.exists():
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        f"Non trovo {nombre}. Carica la cartella datos/ (in Colab: pannello dei file) o esegui il notebook dalla sua cartella.")


def tokenizar(texto):
    """Tokenizzazione della lezione 2: parole con re.findall."""
    return re.findall(r"\w+", texto)

NOMBRE_NLTK = {"spa": "spanish", "ita": "italian"}
SW = {idioma: set(stopwords.words(nombre)) for idioma, nombre in NOMBRE_NLTK.items()}
STEMMER = {idioma: SnowballStemmer(nombre) for idioma, nombre in NOMBRE_NLTK.items()}
NLP = {"spa": nlp_es, "ita": nlp_it}
print("Setup pronto.")

---
## 1. Stopword con criterio

Una **stopword** è una parola molto frequente e di poco contenuto (`el`, `de`, `che`). Toglierla riduce il rumore quando conti le parole. Ma una lista di stopword è una decisione, non una verità: vediamo cosa c'è dentro.

In [ ]:
for idioma in ("spa", "ita"):
    lista = sorted(SW[idioma])
    print(idioma, "→", len(lista), "parole; prime 12:", lista[:12])

In [ ]:
# datos-es
FRASES_NEGACION = {
    "spa": ["no está bien", "la entrega no fue rápida", "sin problemas", "no llegó nada"],
    "ita": ["non è buono", "la consegna non è stata veloce", "senza problemi", "non è arrivato niente"],
}
NEGACIONES = {
    "spa": {"no", "nunca", "jamás", "ni", "sin", "nada", "nadie", "tampoco"},
    "ita": {"non", "mai", "né", "senza", "niente", "nessuno", "nulla", "neanche"},
}
DOMINIO = {"spa": {"pedido", "pedidos"}, "ita": {"ordine", "ordini"}}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for palabra in sorted(NEGACIONES[idioma]):
        filas.append({"lingua": idioma, "parola": palabra, "è nella lista di NLTK?": palabra in SW[idioma]})
tabla_neg = pd.DataFrame(filas)
print(tabla_neg.to_string(index=False))

**Cosa osservi?** Le liste contengono alcune negazioni (`no`, `ni`, `sin`, `nada` in spagnolo; `non` in italiano) ma **non tutte**: `nunca`, `jamás`, `nadie` e `tampoco` non sono nella lista spagnola, e in italiano c'è solo `non`. Le due liste non sono equivalenti: togliere le stopword in spagnolo elimina `sin`, mentre in italiano `senza` resta. Con `spaCy` o con qualunque altra lista la ripartizione sarebbe diversa: **bisogna sempre guardare la lista**.

### 1.1 Quando togliere le stopword cambia il significato

Applichiamo il filtro standard (`quitar_stopwords`) a quattro frasi per lingua. Confronta ciò che resta con ciò che diceva la frase.

In [ ]:
def quitar_stopwords(tokens, idioma):
    """Filtro standard: toglie tutto ciò che è nella lista di NLTK."""
    return [t for t in tokens if t.lower() not in SW[idioma]]

for idioma in ("spa", "ita"):
    for frase in FRASES_NEGACION[idioma]:
        print(f"{frase:36} → {quitar_stopwords(tokenizar(frase), idioma)}")

**Cosa osservi?** `no está bien` resta `['bien']`: un testo che dice il contrario dell'originale. `la entrega no fue rápida` diventa `['entrega', 'rápida']` (anche la parola `fue` è una stopword!) e `sin problemas` (positivo) diventa `['problemas']`, che sembra un reclamo. In italiano `non è buono` resta `['buono']`, ma `senza problemi` conserva `senza`. Per un sacco di parole che si limita a contare, `bien` e `problemas` sono i segnali che il classificatore vedrà: la negazione è sparita.

### 1.2 Una lista personalizzata

La soluzione non è «non usare stopword», ma **costruire la lista con criterio**: partire da quella di NLTK, **conservare le negazioni** e, se il problema lo richiede, **aggiungere parole di dominio** (in un motore di ricerca di ticket, `pedido`/`ordine` compaiono in quasi tutti e aiutano poco a distinguerli).

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def quitar_stopwords_pers(tokens, idioma):
    return [t for t in tokens if t.lower() not in SW_PERS[idioma]]

for idioma in ("spa", "ita"):
    print(idioma, "NLTK:", len(SW[idioma]), "→ personalizzata:", len(SW_PERS[idioma]))
    for frase in FRASES_NEGACION[idioma]:
        print(f"   {frase:36} → {quitar_stopwords_pers(tokenizar(frase), idioma)}")

In [ ]:
# Recensioni: quanti token sono stopword e quante recensioni contengono una negazione della lista di NLTK
resenas = {"spa": pd.read_csv(cargar_datos("resenas_es.csv")), "ita": pd.read_csv(cargar_datos("resenas_it.csv"))}
tickets = {"spa": pd.read_csv(cargar_datos("tickets_es.csv")), "ita": pd.read_csv(cargar_datos("tickets_it.csv"))}

filas = []
for idioma in ("spa", "ita"):
    minus = [[t.lower() for t in tokenizar(x)] for x in resenas[idioma]["texto"]]
    n_tokens = sum(len(l) for l in minus)
    n_stop = sum(t in SW[idioma] for l in minus for t in l)
    negaciones_lista = NEGACIONES[idioma] & SW[idioma]
    con_neg = sum(any(t in negaciones_lista for t in l) for l in minus)
    filas.append({"lingua": idioma, "token": n_tokens, "% stopword": round(100 * n_stop / n_tokens, 1),
                  "recensioni con negazione della lista": con_neg, "su": len(minus)})
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Circa la metà dei token sono stopword (si risparmiano memoria e rumore), ma una parte importante delle recensioni contiene una negazione della lista: sono proprio le recensioni in cui il filtro standard può invertire il senso. Ricorda che il corpus è sintetico e piccolo (*esempio illustrativo*): guarda lo schema, non la cifra esatta.

---
## 2. Stemming con Snowball

Lo **stemming** taglia ogni parola fino a una **radice** applicando regole sui suffissi. Non usa un dizionario, quindi **non garantisce di ottenere una parola vera** (`universidad` → `univers`). È veloce e deterministico. `SnowballStemmer("spanish")` e `SnowballStemmer("italian")` sono quelli di NLTK.

In [ ]:
# datos-es
PALABRAS = {
    "spa": ["universidad", "universo", "generar", "generación", "generoso", "general",
            "organizar", "órgano", "viene", "venir", "fue", "fueron", "ir", "iba", "voy", "bueno", "mejor"],
    "ita": ["università", "universo", "generare", "generazione", "generoso", "generale",
            "organizzare", "organo", "viene", "venire", "fu", "furono", "andare", "andava", "vado", "buono", "migliore"],
}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for palabra in PALABRAS[idioma]:
        filas.append({"lingua": idioma, "forma": palabra, "radice Snowball": STEMMER[idioma].stem(palabra)})
tabla_raices = pd.DataFrame(filas)
print(tabla_raices.to_string(index=False))

### 2.1 Sovra-stemming e sotto-stemming

Due errori tipici, che si vedono raggruppando le parole per radice:

- **Sovra-stemming** (*over-stemming*): parole **diverse** con la **stessa** radice.
- **Sotto-stemming** (*under-stemming*): forme della **stessa** parola con radici **diverse** (soprattutto nei verbi irregolari).

In [ ]:
for idioma in ("spa", "ita"):
    grupos = {}
    for palabra in PALABRAS[idioma]:
        grupos.setdefault(STEMMER[idioma].stem(palabra), []).append(palabra)
    print(idioma)
    for raiz, formas in grupos.items():
        if len(formas) > 1:
            print(f"   stessa radice {raiz!r:12} ← {formas}")
    print("   radici diverse per forme di venire/andare (sotto-stemming):",
          sorted({STEMMER[idioma].stem(p) for p in PALABRAS[idioma][8:16] if p not in ("bueno", "buono")}))

In [ ]:
# Uno stemmer di un'altra lingua rovina il testo: lo stemmer inglese su parole spagnole e italiane
STEMMER_EN = SnowballStemmer("english")
filas = []
for idioma, palabras in (("spa", ["cantaremos", "universidad", "organización", "entregado"]), ("ita", ["canteremo", "università", "organizzazione", "consegnato"])):  # dato-es
    for palabra in palabras:
        filas.append({"lingua": idioma, "forma": palabra, "Snowball della lingua": STEMMER[idioma].stem(palabra), "Snowball inglese": STEMMER_EN.stem(palabra)})
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** `universidad` e `universo` condividono la radice (`univers`), così come `generar`, `generación` e `generoso` (`gener`): sovra-stemming. Invece `viene`, `venir`, `fue`, `iba`, `voy` oppure `andare`, `vado`, `andava` hanno radici diverse: sotto-stemming. E lo stemmer inglese tocca appena le parole spagnole o italiane (o le taglia male): usare lo stemmer sbagliato non dà errore, solo un risultato peggiore.

---
## 3. Lemmatizzazione con spaCy

La **lemmatizzazione** restituisce il **lemma**: la forma da dizionario (`cantaron` → `cantar`, `ordini` → `ordine`). Usa un modello statistico e regole, e **dipende dalla categoria grammaticale** (POS, *part of speech*). Qui ti servono solo quattro etichette universali (UPOS): `NOUN` (sostantivo), `VERB` (verbo), `ADJ` (aggettivo) e `AUX` (ausiliare); la lezione 7 le studia a fondo.

### 3.1 Lo stesso token, due lemmi

In [ ]:
# datos-es
CASOS_POS = {
    "spa": ["Ayer vino mi hermana", "Con la cena tomamos vino tinto", "Yo trabajo mucho", "El trabajo es difícil"],
    "ita": ["Io lavoro molto", "Il lavoro è difficile", "Il vino è buono", "Ho pianto tutta la notte"],
}

In [ ]:
for idioma in ("spa", "ita"):
    for frase in CASOS_POS[idioma]:
        doc = NLP[idioma](frase)
        print(f"{frase:34} →", [(t.text, t.lemma_, t.pos_) for t in doc])

**Cosa osservi?** In spagnolo `vino` è il verbo *venir* in `Ayer vino mi hermana` (lemma `venir`) e la bevanda in `tomamos vino tinto` (lemma `vino`); `trabajo` è il verbo `trabajar` o il sostantivo `trabajo` a seconda della frase. In italiano succede lo stesso con `lavoro`. Un lemmatizzatore che vede solo la forma isolata non può decidere: gli serve la frase. Nota anche un errore del modello piccolo: in `Il vino è buono`, `buono` può uscire etichettato come `VERB` (con spaCy 3.8; può variare secondo la versione): l'etichetta sbaglia, anche se il lemma è corretto. **Non fidarti del POS senza guardarlo.**

### 3.2 Errori tipici del modello piccolo

I modelli `_sm` sono veloci e piccoli, e falliscono in quattro situazioni: **clitici** (`dámelo`, `dammelo`, già visti nella lezione 1), **forme isolate** (una parola sola non dà contesto), **nomi propri** (maiuscole a inizio frase) e **testo rumoroso**. Con spaCy 3.8 e `es_core_news_sm`/`it_core_news_sm` 3.8.0 (possono variare leggermente con altre versioni):

In [ ]:
# datos-es
CASOS_ERROR = {
    "spa": ["Dámelo mañana", "dame el paquete", "Quiero devolverlo hoy", "Miguel de Cervantes vive en Madrid", "vino", "Hablo con Ana"],
    "ita": ["Dammelo domani", "Passamelo domani", "Vorrei restituirlo", "Marco Rossi vive a Milano", "vino", "Io canto ogni sera"],
}

In [ ]:
for idioma in ("spa", "ita"):
    for frase in CASOS_ERROR[idioma]:
        doc = NLP[idioma](frase)
        print(f"{frase:36} →", [(t.text, t.lemma_, t.pos_) for t in doc])

**Cosa osservi?** I clitici danno lemmi strani: `Dámelo` diventa `dámelir`, `devolverlo` diventa `devolver él`, `restituirlo` diventa `restituire lo` e `Dammelo` resta un nome proprio senza lemmatizzare; `Hablo` a inizio frase è preso per nome proprio; una parola sola come `vino` riceve il lemma che il modello ritiene più probabile, senza contesto (in spagnolo, `venir`). Non c'è nessun errore visibile: il lemma è semplicemente dubbio. Per questo conviene **controllare un campione** di ciò che esce prima di fidarsi.

### 3.3 Testo rumoroso: `texto` contro `texto_limpio`

Le recensioni del corpus hanno una versione grezza (`texto`, con maiuscole, accenti persi, allungamenti, abbreviazioni da chat) e una pulita (`texto_limpio`). Confrontiamo i lemmi delle stesse recensioni.

In [ ]:
def lemas_de(texto, idioma):
    """Lemmi in minuscolo dei token alfanumerici di un testo (contesto = il testo intero)."""
    return [t.lemma_.lower() for t in NLP[idioma](texto) if re.fullmatch(r"\w+", t.text)]

for idioma in ("spa", "ita"):
    df = resenas[idioma]
    ruidosas = df[(df["texto"] != df["texto_limpio"]) & (df["texto"].str.len() < 95)].head(2)
    for _, fila in ruidosas.iterrows():
        print(idioma, fila["id"])
        print("   grezzo: ", fila["texto"])
        print("           →", lemas_de(fila["texto"], idioma))
        print("   pulito:", fila["texto_limpio"])
        print("           →", lemas_de(fila["texto_limpio"], idioma))

**Cosa osservi?** Gli allungamenti (`reservaaa`) e i refusi (`cacnelar`, `lascinao`) passano tali e quali nella lista dei lemmi o si lemmatizzano male: la stessa recensione dà liste diverse. La pulizia della lezione 2 **prima** di lemmatizzare non è facoltativa: il lemmatizzatore è buono quanto il testo che riceve.

---
## 4. NLTK contro spaCy: una tabella

Per ogni forma usiamo una frase breve come contesto (spaCy ne ha bisogno) e mostriamo la radice Snowball della forma e il lemma e il POS che dà spaCy **per quel token**. Le cifre di spaCy dipendono dalla versione.

In [ ]:
# datos-es
COMPARACION = {
    "spa": [("cantaron", "Ellos cantaron muy bien"), ("mejores", "Los mejores productos"), ("pedidos", "Los pedidos llegaron tarde"),
            ("entregado", "Me han entregado el paquete"), ("fueron", "Los paquetes fueron entregados"), ("vino", "Ayer vino mi hermana"),
            ("dámelo", "Dámelo mañana"), ("universidad", "La universidad y el universo"), ("Cervantes", "Miguel de Cervantes vive en Madrid")],
    "ita": [("cantarono", "Loro cantarono molto bene"), ("migliori", "I migliori prodotti"), ("ordini", "Gli ordini sono arrivati in ritardo"),
            ("consegnato", "Mi hanno consegnato il pacco"), ("furono", "I pacchi furono consegnati"), ("vino", "Il vino è buono"),
            ("dammelo", "Dammelo domani"), ("università", "L'università e l'universo"), ("Rossi", "Marco Rossi vive a Milano")],
}

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    for forma, frase in COMPARACION[idioma]:
        doc = NLP[idioma](frase)
        token = next(t for t in doc if t.text.lower() == forma.lower())
        filas.append({"lingua": idioma, "forma": forma, "radice Snowball": STEMMER[idioma].stem(forma.lower()),
                      "lemma spaCy": token.lemma_, "POS": token.pos_})
tabla_comp = pd.DataFrame(filas)
print(tabla_comp.to_string(index=False))

**Cosa osservi?** Confronta colonna per colonna. La radice non è mai una parola «bella» (`ped`, `entreg`), ma raggruppa forme correlate; il lemma è una parola vera quando il modello ci azzecca, e sbaglia con i clitici (`dámelo` → `dámelir`) e con forme poco frequenti (`cantarono` non viene lemmatizzato). Lo stemmer, da parte sua, taglia anche i nomi propri (`Cervantes` → `cervant`, `Rossi` → `ross`), che il lemmatizzatore lascia intatti. E le forme irregolari (`fueron`, `furono`) si lemmatizzano bene con spaCy ma restano isolate nello stemmer.

---
## 5. Esperimento guidato: l'effetto sul vocabolario

**Domanda:** quanto si riduce il vocabolario a ogni passo? Misuriamo il numero di **tipi** (parole distinte) e il rapporto **tipi/token** su recensioni e ticket, nelle due lingue, dopo ogni passo: grezzo → minuscolo → senza stopword → stemming → lemmatizzazione. Attenzione: **non misuriamo la qualità di un classificatore** (quella, nella lezione 5), solo la dimensione del vocabolario. Tokenizziamo con `re.findall` come nella lezione 2. Stemming e lemmatizzazione partono dai token senza stopword (sono due alternative, non una dopo l'altra).

In [ ]:
PASOS = ["grezzo", "minuscolo", "senza stopword", "stemming", "lemmatizzazione"]

def medir(textos, idioma):
    """Restituisce {passo: lista di token} per un insieme di testi."""
    brutos = [t for texto in textos for t in tokenizar(texto)]
    minusculas = [t.lower() for t in brutos]
    sin_sw = [t for t in minusculas if t not in SW[idioma]]
    raices = [STEMMER[idioma].stem(t) for t in sin_sw]
    lemas = []
    for doc in NLP[idioma].pipe(list(textos), disable=["ner", "parser"]):
        for t in doc:
            if re.fullmatch(r"\w+", t.text) and t.text.lower() not in SW[idioma]:
                lemas.append(t.lemma_.lower())
    return dict(zip(PASOS, [brutos, minusculas, sin_sw, raices, lemas]))

def resumen(pasos):
    filas = []
    for paso, lista in pasos.items():
        tipos = len(set(lista))
        filas.append({"passo": paso, "tokens": len(lista), "tipi": tipos, "tipi/token": round(tipos / len(lista), 3)})
    return pd.DataFrame(filas)

corpus = {
    ("recensioni", "spa"): resenas["spa"]["texto"], ("recensioni", "ita"): resenas["ita"]["texto"],
    ("tickets", "spa"): tickets["spa"]["texto"], ("tickets", "ita"): tickets["ita"]["texto"],
}
medidas = {clave: medir(textos, clave[1]) for clave, textos in corpus.items()}
for (nombre, idioma), pasos in medidas.items():
    print(nombre, idioma)
    print(resumen(pasos).to_string(index=False))
    print()

In [ ]:
tipos = pd.DataFrame({f"{nombre} ({idioma})": [len(set(l)) for l in pasos.values()] for (nombre, idioma), pasos in medidas.items()}, index=PASOS)
porcentaje = 100 * tipos / tipos.iloc[0]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
tipos.plot(kind="bar", ax=ejes[0], rot=25)
ejes[0].set_ylabel("tipi (parole distinte)")
ejes[0].set_title("Vocabolario dopo ogni passo")
ejes[0].legend(fontsize=8)
porcentaje.plot(marker="o", ax=ejes[1], rot=25)
ejes[1].set_ylabel("% del vocabolario grezzo")
ejes[1].set_title("Vocabolario relativo al grezzo")
ejes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(porcentaje.round(1).to_string())

**Cosa osservi?** Tre cose. **(1)** Passare al minuscolo riduce già il vocabolario tra l'11 % e il 14 %: `Pedido` e `pedido` non sono più due tipi. **(2)** Togliere le stopword cambia poco il numero di tipi (spariscono solo circa 90-110 tipi, anche se sono i token più frequenti, circa la metà del testo). Per questo il rapporto tipi/token **sale**: ci sono meno token, non più varietà. **(3)** Lo stemming riduce il vocabolario più della lemmatizzazione, perché raggruppa di più (comprese parole che non dovrebbe raggruppare: sovra-stemming). Quanta riduzione è utile? Quella che unisce forme della stessa parola; quella che unisce parole diverse è informazione distrutta. Le cifre vengono da corpora sintetici (*esempio illustrativo*).

### 5.1 E con il testo pulito?

Ripetiamo la misurazione dei lemmi su `texto_limpio` (recensioni): se il vocabolario di lemmi cala molto, parte di ciò che sembrava varietà di parole era rumore.

In [ ]:
filas = []
for idioma in ("spa", "ita"):
    bruto = medir(resenas[idioma]["texto"], idioma)
    limpio = medir(resenas[idioma]["texto_limpio"], idioma)
    for paso in PASOS:
        filas.append({"lingua": idioma, "passo": paso, "tipi (grezzo)": len(set(bruto[paso])), "tipi (pulito)": len(set(limpio[paso]))})
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Il testo pulito ha meno tipi a ogni passo: il rumore (maiuscole, refusi, allungamenti) gonfia il vocabolario e riduce l'utilità della lemmatizzazione. Pulire prima (lezione 2) e normalizzare dopo (questa lezione) è l'ordine naturale.

---
## 6. Guida alla decisione: quale preprocessing per quale compito

Non esiste un preprocessing universale. Questa tabella riassume una guida ragionevole; si verifica con i dati, non con la fede.

In [ ]:
guia = pd.DataFrame([
    ("Ricerca (lezione 11)", "facoltative; con criterio", "utile (lemma o radice)", "Conviene che la query e i documenti si normalizzino allo stesso modo."),
    ("Classificazione con TF-IDF (lezione 5)", "facoltative (TF-IDF già pesa)", "prova e confronta", "Si decide con la validazione, non a occhio."),
    ("Sentiment con negazioni (lezione 6)", "conserva le negazioni", "con cautela", "`non buono` non è `buono`."),
    ("Riconoscimento di entità (lezione 7)", "no", "no", "Servono maiuscole e contesto."),
    ("Generazione di testo (lezione 8)", "no", "no", "Il testo generato deve essere testo completo."),
], columns=["compito", "stopword", "stemming / lemmatizzazione", "perché"])
print(guia.to_string(index=False))

> **Regola pratica:** applica **esattamente lo stesso** preprocessing in addestramento e in test (e in produzione). Se lo cambi tra i due, il modello vede parole che non conosce. Come evitare la fuga di informazione costruendolo dentro una `Pipeline`, nella lezione 5.

---
## Riepilogo e passo successivo

- Le **stopword** di NLTK includono alcune negazioni e non altre; toglierle cambia il significato di frasi come `no está bien` / `non è buono`. Usa una **lista propria**: conserva le negazioni e aggiungi parole di dominio se serve.
- Lo **stemming** (Snowball) taglia per regole: veloce, senza garanzia di parola vera, con sovra-stemming e sotto-stemming.
- La **lemmatizzazione** (spaCy) dà forme da dizionario e dipende dal POS; i modelli piccoli sbagliano con clitici, forme isolate, nomi propri e testo rumoroso.
- L'effetto sul **vocabolario** si misura, non si suppone: minuscolo e stemming riducono molto; togliere le stopword riduce poco i tipi.
- Bisogna **applicare lo stesso** preprocessing in addestramento e test.

Continua con la lezione 4: Rappresentazioni sparse (sacco di parole e TF-IDF).